# Graph phase oracles: test the predicate before claiming a clique solver

Build a five-vertex synthetic graph, compare pairwise phase constructions with an exact fixed-cardinality clique predicate, and measure one Grover search experiment. The central skill is falsifying an oracle claim using truth tables and statevector invariants. This is a small verification experiment, not a scalable clique algorithm.

**Prerequisites:** Python 3.12 or later and the tested environment in the [collection README](../../../README.md). Install the [root requirements](../../../requirements.txt) into an isolated environment before opening this notebook; select that environment's kernel. Run all cells from a fresh kernel in order. This notebook makes no network calls, reads no external files, and writes no artifacts by default.

**Reproducibility:** all examples and data here are synthetic. Simulator and sampling seeds are explicit. Assertions distinguish mathematical identities, finite-shot checks, and integration contracts. Approximate comparisons state their tolerance.


Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../../AI_DISCLOSURE.md), [license](../../../LICENSE), and [attribution](../../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Define the combinatorial question

A basis string selects vertices with bit value one. We seek subsets of **exactly three** vertices whose induced subgraph has every possible edge. If $v(x)$ counts selected nonedges, a clique requires $v(x)=0$ **and** $|x|=3$. A product of CZ gates on nonedges instead computes $(-1)^{v(x)}$: it detects odd parity, not the absence of violations.


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit
from qiskit.circuit.library import DiagonalGate
from qiskit.quantum_info import Statevector


def show_plot():
    import io

    from IPython.display import Image, display

    buffer = io.BytesIO()
    plt.gcf().savefig(buffer, format="png", dpi=120, bbox_inches="tight")
    display(Image(data=buffer.getvalue()))
    plt.close()


n, k = 5, 3
G = nx.Graph()
G.add_nodes_from(range(n))
G.add_edges_from([(0, 1), (0, 2), (1, 2), (2, 3), (3, 4)])
nonedges = sorted(nx.non_edges(G))


def violations(x):
    return sum(((x >> a) & 1) and ((x >> b) & 1) for a, b in nonedges)


def is_target(x):
    return x.bit_count() == k and violations(x) == 0


marked = [x for x in range(2**n) if is_target(x)]
assert marked == [0b00111]
false_parity_accepts = [
    x for x in range(2**n) if x.bit_count() == k and violations(x) > 0 and violations(x) % 2 == 0
]
assert false_parity_accepts  # A constructive counterexample to parity-as-validity.
print("Exact target:", [f"{x:05b}" for x in marked])
print("Invalid size-3 strings with even violations:", [f"{x:05b}" for x in false_parity_accepts])
nx.draw_networkx(G, pos=nx.circular_layout(G), node_color="#d6eaf8")
plt.axis("off")
show_plot()

## 2. Compare two phase heuristics with the exact oracle

The weighted heuristic assigns each missing pair angle $\pi/d_b$, where $d_b$ is the number of nonedges incident on its higher-numbered endpoint. This is deliberately an experimental normalization; it depends on labeling and has no clique-validity proof. Both heuristics are diagonal, so neither changes the probabilities of a uniform superposition until another mixing operation is applied. The exact oracle flips only satisfying strings; we synthesize it from its complete truth table, paying exponential classical work.


In [ ]:
dimension = 2**n
exact_signs = np.array([-1 if is_target(x) else 1 for x in range(dimension)], dtype=complex)
parity = QuantumCircuit(n)
weighted = QuantumCircuit(n)
missing_degree = dict(nx.complement(G).degree())
for a, b in nonedges:
    parity.cz(a, b)
    weighted.cp(np.pi / missing_degree[b], a, b)
exact = QuantumCircuit(n)
exact.append(DiagonalGate(exact_signs), range(n))


def distribution(oracle, mixing=False):
    qc = QuantumCircuit(n)
    qc.h(range(n))
    qc.compose(oracle, inplace=True)
    if mixing:
        qc.h(range(n))
    return Statevector.from_instruction(qc).probabilities()


for oracle in (parity, weighted, exact):
    assert np.allclose(distribution(oracle), np.full(dimension, 1 / dimension), atol=1e-12)
rows = []
for label, oracle in [
    ("CZ parity", parity),
    ("Weighted heuristic", weighted),
    ("Exact predicate", exact),
]:
    probs = distribution(oracle, mixing=True)
    rows.append(
        {"construction": label, "target_probability_after_H": sum(probs[x] for x in marked)}
    )
print(pd.DataFrame(rows))

## 3. Add amplitude amplification to the exact predicate

An oracle alone does not search. The diffusion operator is $2|s\rangle\langle s|-I$, where $|s\rangle$ is uniform. With one marked state among 32, $\theta=\arcsin(\sqrt{1/32})$ and ideal success after $r$ iterations is $\sin^2((2r+1)\theta)$. Our circuit uses the negative of the diffusion operator, which changes only global phase. Compare the measured quantity with this independent prediction.


In [ ]:
theta = np.arcsin(np.sqrt(len(marked) / dimension))
iterations = int(np.round(np.pi / (4 * theta) - 0.5))
search = QuantumCircuit(n)
search.h(range(n))
reflection = np.ones(dimension)
reflection[0] = -1
for _ in range(iterations):
    search.compose(exact, inplace=True)
    search.h(range(n))
    search.append(DiagonalGate(reflection), range(n))
    search.h(range(n))
probabilities = Statevector.from_instruction(search).probabilities()
actual = sum(probabilities[x] for x in marked)
predicted = np.sin((2 * iterations + 1) * theta) ** 2
assert abs(actual - predicted) < 1e-11 and actual > 0.90
print({"iterations": iterations, "success_probability": actual, "analytic_prediction": predicted})
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(range(dimension), probabilities)
ax.set(
    xlabel="Selected vertex bitmask (integer)",
    ylabel="Probability",
    title="Exact oracle plus Grover diffusion",
)
show_plot()

## Limits, cleanup, and next steps

Enumerating the target set and synthesizing `DiagonalGate` already solve the small instance classically. The result verifies circuit semantics; it provides no end-to-end speedup claim. Weighted phases are not automatically penalty thresholds, and vertex relabeling can change the weighted heuristic. A scalable algorithm needs a reversible predicate circuit and a complete cost model including state preparation and oracle synthesis.

No files or services need cleanup. An extension is to build a reversible violation counter and compare its ancilla and two-qubit gate cost with truth-table synthesis.

Primary references: [Grover's original search paper](https://arxiv.org/abs/quant-ph/9605043), [DiagonalGate API](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.DiagonalGate), [NetworkX clique algorithms](https://networkx.org/documentation/stable/reference/algorithms/clique.html).
